### A function that accepts a variable (e.g. “RH”) and year range (From, To) and dynamical-downloading-locally-No-hard-coded-path-needed ---> scrapes the dataset website to discover the relevant files instead of relying on hardcoded paths.¶

In [1]:
# Q1: load libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib3.util.retry import Retry 
from requests.adapters import HTTPAdapter 

import os
import xarray as xr
import requests
#import hashlib  # Import hashlib module

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [3]:
def get_ncep_data(variable, year_start, year_end):
    
    base_url = "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis.dailyavgs/surface/" # Change as needed
    
    # Setup request object ---> so we work with the same session and don't have to re-connect 
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))

    print(f"Connecting to NOAA server...")
    response = session.get(base_url, timeout=30)
    response.raise_for_status() 
        
    soup = BeautifulSoup(response.text, 'html.parser')
    data_records = []

    for link in soup.find_all('a', href=True):
        href = link['href']
        
        # 1. First check: Is it a NetCDF file and does it contain our variable?
        if href.endswith('.nc') and variable in href:
            
            # 2. Extract the year using a simpler search
            year_match = re.search(r'(\d{4})', href)
            
            if year_match:
                year = int(year_match.group(1))
                
                # 3. Filter by range ---> our final gatekeeper: Checks if the years are in the range
                if year_start <= year <= year_end:
                    data_records.append({
                        'filename': href, 
                        'download_url': base_url + href
                    })

    return pd.DataFrame(data_records)

In [4]:
# --- Test: Call the function above --- #

variable = 'rhum' #'rhum' #"air"
start_year = 1981
end_year = 1982 # if we give say 2027-which does not exist, it drops that & continues appending to the df
df = get_ncep_data(variable, start_year, end_year)
print(df)
print(df.shape)


Connecting to NOAA server...
              filename                                       download_url
0  rhum.sig995.1981.nc  https://downloads.psl.noaa.gov/Datasets/ncep.r...
1  rhum.sig995.1982.nc  https://downloads.psl.noaa.gov/Datasets/ncep.r...
(2, 2)


In [5]:
# --- 2. Download the data using df --- #

# i. Environment Setup: Create a directory if it does not yet exist 
output_dir = 'ncep_data_daily_air'
os.makedirs(output_dir, exist_ok=True)

successful_downloads = []
failed_downloads = []

print(f"Starting to acquire {len(df)} files based on the above data frame in the code block (Q1)...")

# ii. Iterate directly through the DataFrame from F1
for index, row in df.iterrows():
    url = row['download_url']
    filename = row['filename']
    local_file_path = os.path.join(output_dir, filename)

    print(f"\n[{index + 1}/{len(df)}] Processing: {filename}") # progress bar for the user

    # Check (Skip if already downloaded): e.g., If air.1982.nc exists, it skips- check takes short time, downloading takes 10-30 secs
    if os.path.exists(local_file_path):   
        print(f" -> File already exists locally. Skipping download.")
        successful_downloads.append(local_file_path) #gives final summary report of 45 or the existing files 
        continue #if file exists, it stops & skips the rest of the code below

    try:
        # Stream the download to handle large NetCDF files safely
        print(f" --> Downloading from NOAA...")
        r = requests.get(url)# makes a connection & sends a request to the NOAA server asking for the specific file (e.g., air.1982.nc). #, timeout=30, stream=True) # 
        r.raise_for_status() # catches the error--> else the script might "successfully" download an error message instead of real data.

        # Writing binary data (nc) to the file, instead of grabbing the eintire file, the code reaches out
        #the server in chunks of 1mb
        with open(local_file_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024): # 1MB chunks
                if chunk:
                    f.write(chunk) # writes the data locally 
        print("Done!")

        # --- Extra --- #
        # iii. Validation: Verify if it is a valid NetCDF and contains the variable
        with xr.open_dataset(local_file_path) as ds:
            if variable in ds.data_vars:
                successful_downloads.append(local_file_path)
            else:
                print(f"  -> WARNING: Variable '{variable}' not found in file!")
                failed_downloads.append(filename)

    except Exception as e:
        print(f"  -> ERROR: {e}")
        # I added this to clean up partial/corrupted downloads
        if os.path.exists(local_file_path):
            os.remove(local_file_path)
        failed_downloads.append(filename)

# --- Summary Report ---
print("FINAL DOWNLOAD SUMMARY")
print(f"Successfully processed: {len(successful_downloads)}")
print(f"Failed/Missing:         {len(failed_downloads)}")
if failed_downloads:
    print(f"Check these years: {failed_downloads}")

Starting to acquire 2 files based on the above data frame in the code block (Q1)...

[1/2] Processing: rhum.sig995.1981.nc
 --> Downloading from NOAA...
Done!

[2/2] Processing: rhum.sig995.1982.nc
 --> Downloading from NOAA...
Done!
FINAL DOWNLOAD SUMMARY
Successfully processed: 2
Failed/Missing:         0
